In [2]:
import requests  # [+] HTTP 요청
from bs4 import BeautifulSoup  # [+] 웹스크래핑
import pandas as pd

In [3]:
# 1. User-Agent 설정
# 우리가 실제 크롬 브라우저를 사용하는 일반 사용자인 것처럼 서버를 속이는 '위장 신분증' 역할
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

search_keyword = '이재명'  # [+] 검색 키워드
url = f'https://search.naver.com/search.naver?where=news&query={search_keyword}'  # 검색한 웹페이지 주소

In [7]:
# 2. 페이지 요청 및 파싱
res = requests.get(url, headers=headers)  # [+] 네이버 서버에 페이지 정보 요청(GET)
res.raise_for_status() # 서버가 응답을 잘 주었는지 확인(정상 200, 페이지 없음 404 등)
soup = BeautifulSoup(res.text, 'html.parser')  # [+] BeautifulSoup 객체 생성

In [21]:
# 3. 질문자님이 찾은 클래스로 타겟팅 (띄어쓰기를 '.'으로 변경)
# [+] 찾고자 하는 HTML 태그 정의
target_class = 'span.sds-comps-text.sds-comps-text-ellipsis.sds-comps-text-ellipsis-1.sds-comps-text-type-headline1'
title_spans = soup.select(target_class) # [+] 조건에 맞는 모든 태그를 찾아 리스트로 저장

extracted_data = []

In [22]:
title_spans

[<span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1"><mark>이재명</mark> 대통령 "세종 집무실, 임기 내 사용"…신속 공사 지시</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">'구속 위기' 전한길 "<mark>이재명</mark> 정권이 감당할 수 있나"</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1"><mark>이재명</mark> 대통령, '이스라엘 비판' 겨냥 야당에 "오목 좀 둔다고 판에 엎...</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1"><mark>이재명</mark> 대통령, '고유가 피해지원금, 지자체 비인권적 행태 반복되지 않...</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1"><mark>이재명</mark> 대통령 "대한민국 국민, 웬만한 사람은 다 전과 있어"</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">민주, 선서 안한 박상용에 또 “퇴장”… 국힘 “<mark>이재명</mark> 죄 지

In [23]:
title_spans[0].text.strip()

'이재명 대통령 "세종 집무실, 임기 내 사용"…신속 공사 지시'

In [24]:
# 4. 데이터 추출
for span in title_spans:
    # [+] 기사 제목 추출
    title = span.text.strip()
    
    # [+] 기사 링크 추출: 제목(span)을 클릭 가능하게 만드는 부모 <a> 태그를 역추적하여 찾음
    parent_a_tag = span.find_parent('a')
    link = parent_a_tag['href'] if parent_a_tag and parent_a_tag.has_attr('href') else "링크 없음"
    
    extracted_data.append({
        '제목': title,
        '링크': link
    })

In [25]:
extracted_data

[{'제목': '이재명 대통령 "세종 집무실, 임기 내 사용"…신속 공사 지시',
  '링크': 'https://www.sidae.com/article/2026041414085065579'},
 {'제목': '\'구속 위기\' 전한길 "이재명 정권이 감당할 수 있나"',
  '링크': 'http://mbn.mk.co.kr/pages/news/newsView.php?category=mbn00009&news_seq_no=5187842'},
 {'제목': '이재명 대통령, \'이스라엘 비판\' 겨냥 야당에 "오목 좀 둔다고 판에 엎...',
  '링크': 'https://imnews.imbc.com/news/2026/politics/article/6815015_36911.html'},
 {'제목': "이재명 대통령, '고유가 피해지원금, 지자체 비인권적 행태 반복되지 않...",
  '링크': 'https://www.newsis.com/view/NISX20260414_0003590460'},
 {'제목': '이재명 대통령 "대한민국 국민, 웬만한 사람은 다 전과 있어"',
  '링크': 'https://www.hankyung.com/article/2026041497447'},
 {'제목': '민주, 선서 안한 박상용에 또 “퇴장”… 국힘 “이재명 죄 지우기 특위...',
  '링크': 'https://www.munhwa.com/article/11582175?ref=naver'},
 {'제목': '전한길 "美백악관이 날 초청했다…이재명 정권 감당할 수 있나" 주장',
  '링크': 'https://view.asiae.co.kr/article/2026041408263561426'},
 {'제목': '이재명 정부 2조원 뿌린다, 주인공은?…GPU 사업에 대기업 총출동',
  '링크': 'https://www.mk.co.kr/article/12016222'},
 {'제목': '질문하는 이재명 대통령',
  '링크': 'https://www.yna.co.kr/view/PYH

In [26]:
# 5. [+] DataFrame으로 변환 및 확인
df = pd.DataFrame(extracted_data)
print(f"총 {len(df)}개의 기사를 추출했습니다.\n")
display(df)

총 10개의 기사를 추출했습니다.



,제목,링크
0,"이재명 대통령 ""세종 집무실, 임기 내 사용""…신속 공사 지시",https://www.sidae.com/article/2026041414085065579
1,"'구속 위기' 전한길 ""이재명 정권이 감당할 수 있나""",http://mbn.mk.co.kr/pages/news/newsView.php?ca...
2,"이재명 대통령, '이스라엘 비판' 겨냥 야당에 ""오목 좀 둔다고 판에 엎...",https://imnews.imbc.com/news/2026/politics/art...
3,"이재명 대통령, '고유가 피해지원금, 지자체 비인권적 행태 반복되지 않...",https://www.newsis.com/view/NISX20260414_00035...
4,"이재명 대통령 ""대한민국 국민, 웬만한 사람은 다 전과 있어""",https://www.hankyung.com/article/2026041497447
5,"민주, 선서 안한 박상용에 또 “퇴장”… 국힘 “이재명 죄 지우기 특위...",https://www.munhwa.com/article/11582175?ref=naver
6,"전한길 ""美백악관이 날 초청했다…이재명 정권 감당할 수 있나"" 주장",https://view.asiae.co.kr/article/2026041408263...
7,"이재명 정부 2조원 뿌린다, 주인공은?…GPU 사업에 대기업 총출동",https://www.mk.co.kr/article/12016222
8,질문하는 이재명 대통령,https://www.yna.co.kr/view/PYH2026041404700001...
9,"'이재명 대통령 최측근' 김용 ""보궐선거 출마 예정"" [뉴시스Pic]",https://www.newsis.com/view/NISX20260413_00035...
